<img height="100" src="https://i.postimg.cc/gjptBxF4/logo-gas-removebg-preview.png" width="250"/>

| Country        | States                                                       | Producing Regions                                                                        | Productivity Data | Soil File | Average Cycle |
|----------------|--------------------------------------------------------------|------------------------------------------------------------------------------------------|-------------------|-----------|---------------|
| United States  | Iowa, Illinois, Nebraska, Minnesota, Indiana                 | Corn Belt (IA, IL, IN); East/Center of NE; South of MN                                   | USDA              | EC6       | Apr – Nov     |
| China          | Heilongjiang, Jilin, Nei Mongol, Shandong, Henan             | Northeast and North China Plains                                                         | NBS               | EC6       | Apr – Oct     |
| Brazil         | Mato Grosso, Paraná, Goiás, Mato Grosso do Sul, Minas Gerais | MT (Mid-North), PR (West), GO (South), MS (Southwest), MG (Triangle)                     | SIDRA-IBGE        | EC3       | Jan – Sep     |
| European Union | France, Romania, Poland, Hungary, Italy                      | FRA (N. Aquitaine), ROM (South), POL (Lower Silesia), HUN (Great Plain), ITA (Po Valley) | AGRI4CAST         | EC2       | Mar – Dec     |
| Argentina      | Córdoba, Buenos Aires, Santa Fé, Santiago del Estero         | Core Zone (North BA, South SF, Center CD); Southeast SDE                                 | BC EXPLORER       | EC6       | Sep – Aug     |
| India          | Karnataka, Madhya Pradesh, Bihar, Tamil Nadu, Telangana      | Ballari-KA, Chhindwara-MP, "Corn Zone"-BI                                                | DES               | EC4       | Mar – Dec     |
| Mexico         | Sinaloa, Jalisco, Michoacán, Guerrero, Chiapas               | Sinaloa Valleys; Ciénega/Altos Regions (Jalisco)                                         | DGSIAP            | EC4       | Apr – Feb     |

Soil File Legend:

* EC2: medium texture soils
* EC3: medium-fine soils
* EC4: fine soils
* EC6: fine and permeable soils <br>
The average cycle comprises the period from sowing to harvest.

In [ ]:
import gc
import io
import os
import shutil
import stat
import warnings
import zipfile

import earthaccess
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import unicodedata
from matplotlib.patches import FancyArrowPatch
from matplotlib.ticker import FuncFormatter
from pyhdf.SD import SD, SDC
from pyproj import Transformer
from shapely.geometry import Point
from shapely.geometry import Polygon

warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import os
df_final = pd.read_excel(os.path.join(os.getcwd(), "inputs","data","coordinates.xlsx"))
df_final.shape

In [ ]:
df_final = df_final.drop_duplicates(subset=['lat','lon'])
df_final.shape

In [ ]:
# Atribuindo COD_IDS para municipios do Brasil
try:
    # --- Passo 1: Carregar os data do Excel e criar o GeoDataFrame ---

    # Define o caminho do arquivo de entrada
    input_xlsx_path = os.path.join(os.getcwd(), 'inputs', 'data', 'coordinates.xlsx')

    points_df = pd.read_excel(input_xlsx_path)

    # Agora, crie o GeoDataFrame a partir do DataFrame carregado
    points_gdf = gpd.GeoDataFrame(
        points_df,
        geometry=gpd.points_from_xy(points_df.lon, points_df.lat),
        crs="EPSG:4326")

    # --- Passo 2: Baixar o mapa de fronteiras dos municípios ---
    print("Carregando mapa de municípios do Brasil...")
    url_municipios = "https://raw.githubusercontent.com/tbrugz/geodata-br/master/geojson/geojs-100-mun.json"
    municipios_gdf = gpd.read_file(url_municipios)
    # Renomeia a coluna 'id' para corresponder a possíveis data futuros
    municipios_gdf.rename(columns={'id': 'cod_municipio'}, inplace=True)
    print("Mapa carregado.")

    # --- Passo 3: Executar a junção espacial ---
    print("Associando pontos aos municípios...")
    # Realiza a junção para encontrar em qual município cada ponto está
    pontos_com_municipio = gpd.sjoin(points_gdf, municipios_gdf, how="inner", predicate='intersects')
    print("Associação concluída.")

    # --- Passo 4: Limpar e salvar o resultado ---

    print("\nLimpando colunas desnecessárias...")
    # MOVIDO PARA CÁ: Esta operação agora está protegida dentro do bloco 'try'
    pontos_com_municipio.drop(columns=['description', 'index_right', 'geometry'], inplace=True, errors='ignore')
    # Adicionado errors='ignore' para não dar erro se uma das colunas não existir

    # Define o caminho do arquivo de saída (assumindo que seja o mesmo do input)
    output_xlsx_path = input_xlsx_path

    # MOVIDO PARA CÁ: Salvar o arquivo também deve estar no 'try'
    print(f"Salvando resultados na aba 'SIDRA-ids' do arquivo '{os.path.basename(output_xlsx_path)}'...")
    with pd.ExcelWriter(output_xlsx_path, mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
      pontos_com_municipio.to_excel(writer, sheet_name='SIDRA-ids', index=False)

    print("\nProcesso concluído com sucesso!")
    print("\nAmostra do resultado final:")
    print(pontos_com_municipio[['country', 'lat', 'lon', 'name']].head())


except FileNotFoundError:
    print(f"\nERRO: O arquivo de entrada não foi encontrado no caminho especificado. Verifique o caminho.")
except Exception as e:
    print(f"\nOcorreu um erro durante o processo: {e}")

In [ ]:
df_SIDRA = pd.read_excel(os.path.join(os.getcwd(), "inputs","data","coordinates.xlsx"), sheet_name="SIDRA-ids")